# Train Vision Transformer (ViT-B/16) on 100,000 Bird Images
### Run this notebook on **Google Colab** (free GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mithunonline/EyeM/blob/master/notebooks/Train_ViT_Birds_Colab.ipynb)

**Dataset:** 525 Bird Species (~89,885 training images) from Kaggle  
**Model:** ViT-B/16 fine-tuned using HuggingFace Trainer API  
**Output:** `vit_birds_finetuned/` saved to Google Drive

---
## Step 0: Before You Start
1. Go to **Runtime → Change runtime type → GPU (T4 or A100)**
2. Get your Kaggle API key: kaggle.com → Account → API → Create New Token
3. Run cells top to bottom

> **Note:** ViT-B/16 (~86M params) is larger than EfficientNet-B0 (~5M). Training takes longer (~3-4× more time per epoch). Recommended: use A100 if available for faster training.

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → GPU')

In [ ]:
!pip install -q kaggle transformers[torch] accelerate evaluate scikit-learn

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/EyeM_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Models will be saved to: {SAVE_DIR}')

## Step 2: Upload Kaggle API Key

In [ ]:
from google.colab import files
print('Upload kaggle.json (from kaggle.com → Account → API → Create New Token)')
uploaded = files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API key configured.')

## Step 3: Download the 525 Bird Species Dataset

In [ ]:
!kaggle datasets download -d gpiosenka/100-bird-species -p /content/birds --unzip

import os, pathlib
DATA_ROOT = '/content/birds'
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VALID_DIR = os.path.join(DATA_ROOT, 'valid')
TEST_DIR  = os.path.join(DATA_ROOT, 'test')

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)
label2id    = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label    = {i: c for i, c in enumerate(CLASS_NAMES)}

print(f'Bird species: {NUM_CLASSES}')
print(f'Sample:       {CLASS_NAMES[:5]}')

## Step 4: Build Dataset for HuggingFace Trainer

In [ ]:
from torch.utils.data import Dataset
from transformers import ViTImageProcessor
from PIL import Image
import os, glob, numpy as np

MODEL_ID  = 'google/vit-base-patch16-224'
processor = ViTImageProcessor.from_pretrained(MODEL_ID)
print(f'Processor loaded: {MODEL_ID}')


class BirdDataset(Dataset):
    """
    Custom PyTorch Dataset that loads bird images from disk
    and preprocesses them for ViT.
    """
    def __init__(self, root_dir, class_names, label2id, augment=False):
        self.samples    = []
        self.label2id   = label2id
        self.augment    = augment
        self.processor  = processor

        # Collect all image paths
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.png']:
                for img_path in glob.glob(os.path.join(class_dir, ext)):
                    self.samples.append((img_path, label2id[class_name]))

        print(f'Dataset loaded: {len(self.samples):,} images, {len(class_names)} classes')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')

        if self.augment:
            # Manual augmentation before ViT processor
            import torchvision.transforms.functional as TF
            import random
            # Random horizontal flip
            if random.random() > 0.5:
                image = TF.hflip(image)
            # Random rotation ±15°
            angle = random.uniform(-15, 15)
            image = TF.rotate(image, angle)
            # Random brightness/contrast
            image = TF.adjust_brightness(image, random.uniform(0.8, 1.2))
            image = TF.adjust_contrast(image,   random.uniform(0.8, 1.2))

        # ViT processor: resize to 224×224 + normalize
        inputs = self.processor(images=image, return_tensors='pt')
        pixel_values = inputs['pixel_values'].squeeze(0)  # (3, 224, 224)

        return {'pixel_values': pixel_values, 'labels': label}


train_dataset = BirdDataset(TRAIN_DIR, CLASS_NAMES, label2id, augment=True)
val_dataset   = BirdDataset(VALID_DIR, CLASS_NAMES, label2id, augment=False)
test_dataset  = BirdDataset(TEST_DIR,  CLASS_NAMES, label2id, augment=False)

## Step 5: Load ViT Model with Custom Classification Head

In [ ]:
from transformers import ViTForImageClassification
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pretrained ViT with a new classification head for NUM_CLASSES bird species
# ignore_mismatched_sizes=True: allows replacing the 1000-class head with our NUM_CLASSES head
vit_model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # Replace 1000-class head with bird-class head
)
vit_model = vit_model.to(device)

total_params     = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f'Model loaded: {MODEL_ID}')
print(f'Output classes: {NUM_CLASSES}')
print(f'Total params:   {total_params:,}')
print(f'Trainable:      {trainable_params:,}')

## Step 6: Configure HuggingFace Trainer

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score
import numpy as np

OUTPUT_DIR = os.path.join(SAVE_DIR, 'vit_birds_checkpoint')

# ── Training configuration ────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training schedule
    num_train_epochs=15,
    per_device_train_batch_size=32,       # 32 images per GPU batch
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,        # Effective batch = 32×2 = 64

    # Learning rate
    learning_rate=2e-5,                   # Small LR for fine-tuning a pretrained model
    weight_decay=0.01,                    # L2 regularization
    warmup_ratio=0.1,                     # 10% of steps for LR warmup
    lr_scheduler_type='cosine',           # Cosine decay after warmup

    # Evaluation & saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,          # Restore best checkpoint at end
    metric_for_best_model='accuracy',
    greater_is_better=True,

    # Performance
    fp16=torch.cuda.is_available(),       # Mixed precision on GPU
    dataloader_num_workers=2,
    remove_unused_columns=False,

    # Logging
    logging_dir=os.path.join(SAVE_DIR, 'logs'),
    logging_steps=100,
    report_to='none',                     # Disable wandb/tensorboard

    # Save storage
    save_total_limit=2,                   # Keep only last 2 checkpoints
)


def compute_metrics(eval_pred):
    """
    Called by Trainer after each validation epoch.
    Returns accuracy score.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc}


trainer = Trainer(
    model=vit_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print('Trainer configured.')
print(f'  Epochs:          {training_args.num_train_epochs}')
print(f'  Batch size:      {training_args.per_device_train_batch_size} × {training_args.gradient_accumulation_steps} acc steps = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps} effective')
print(f'  Learning rate:   {training_args.learning_rate}')
print(f'  FP16:            {training_args.fp16}')
print(f'  Save dir:        {OUTPUT_DIR}')

## Step 7: Train the Model

In [ ]:
import time
print('Starting ViT fine-tuning...')
print('This will take approximately 2-4 hours on a T4 GPU.')
print('='*65)

start_time = time.time()
train_result = trainer.train()
elapsed = (time.time() - start_time) / 3600

print(f'\nTraining complete in {elapsed:.2f} hours')
print(f'Training loss: {train_result.training_loss:.4f}')

## Step 8: Evaluate on Validation & Test Sets

In [ ]:
# Validation evaluation
val_results = trainer.evaluate(val_dataset)
print(f'Validation Accuracy: {val_results["eval_accuracy"]*100:.2f}%')

# Test evaluation
test_results = trainer.evaluate(test_dataset, metric_key_prefix='test')
print(f'Test Accuracy:       {test_results["test_accuracy"]*100:.2f}%')

## Step 9: Plot Training History

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history

# Extract epoch-level metrics
train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs  = [l for l in log_history if 'eval_accuracy' in l]

eval_epochs   = [l['epoch']            for l in eval_logs]
eval_acc      = [l['eval_accuracy']*100 for l in eval_logs]
eval_loss     = [l['eval_loss']         for l in eval_logs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(eval_epochs, eval_acc, 'g-o', linewidth=2, label='Val Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('ViT-B/16 — Validation Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_epochs, eval_loss, 'r-o', linewidth=2, label='Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('ViT-B/16 — Validation Loss', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'vit_birds_training_curves.png'), dpi=150)
plt.show()
print(f'Best Val Accuracy: {max(eval_acc):.2f}%')

## Step 10: Export Model for Streamlit App

In [ ]:
import json

# Save the fine-tuned ViT model in HuggingFace format
vit_export_dir = os.path.join(SAVE_DIR, 'vit_birds_finetuned')
trainer.save_model(vit_export_dir)
processor.save_pretrained(vit_export_dir)

# Also save a lightweight .pth checkpoint for the Streamlit app
torch.save({
    'model_state_dict': vit_model.state_dict(),
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'architecture': 'vit-base-patch16-224',
    'val_acc': val_results['eval_accuracy'] * 100,
    'test_acc': test_results['test_accuracy'] * 100,
    'training_images': len(train_dataset),
}, os.path.join(SAVE_DIR, 'vit_birds_finetuned.pth'))

# Save class names
with open(os.path.join(SAVE_DIR, 'bird_class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print(f'ViT model saved to: {vit_export_dir}')
print(f'Checkpoint:         {os.path.join(SAVE_DIR, "vit_birds_finetuned.pth")}')
print(f'\nFinal Results:')
print(f'  Classes:         {NUM_CLASSES} bird species')
print(f'  Training images: {len(train_dataset):,}')
print(f'  Val Accuracy:    {val_results["eval_accuracy"]*100:.2f}%')
print(f'  Test Accuracy:   {test_results["test_accuracy"]*100:.2f}%')
print(f'\nDownload vit_birds_finetuned.pth and bird_class_names.json')
print('and place them in the EyeM/models/ directory.')

## Step 11: Attention Map on a Bird Image (ViT Interpretability)

In [ ]:
import urllib.request, io
import torch.nn.functional as F

BIRD_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/4c/Starling_on_the_ground.jpg/640px-Starling_on_the_ground.jpg'

try:
    with urllib.request.urlopen(BIRD_URL) as r:
        test_img = Image.open(io.BytesIO(r.read())).convert('RGB')

    inputs = processor(images=test_img, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = vit_model(**inputs, output_attentions=True)

    logits = outputs.logits
    probs  = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    top5   = probs.argsort()[::-1][:5]

    print('Top-5 Predictions:')
    for i in top5:
        print(f'  {CLASS_NAMES[i]:40s} {probs[i]*100:.2f}%')

    # Attention map
    attn      = outputs.attentions[-1].squeeze(0)   # (12, 197, 197)
    cls_attn  = attn[:, 0, 1:].mean(dim=0).cpu().numpy()  # (196,)
    attn_map  = cls_attn.reshape(14, 14)
    attn_map  = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    attn_up   = F.interpolate(
        torch.tensor(attn_map).unsqueeze(0).unsqueeze(0),
        size=(224, 224), mode='bilinear', align_corners=False
    ).squeeze().numpy()

    img224   = np.array(test_img.resize((224, 224)))
    heatmap  = plt.cm.hot(attn_up)[:, :, :3]
    blended  = np.clip(0.55 * img224 / 255.0 + 0.45 * heatmap, 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img224);  axes[0].set_title('Test Bird Image');   axes[0].axis('off')
    axes[1].imshow(blended); axes[1].set_title(f'ViT Attention — {CLASS_NAMES[top5[0]]}'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f'Test failed: {e}. Model is ready — test with your own images.')

## Summary

| Metric | Value |
|---|---|
| Architecture | ViT-B/16 (fine-tuned) |
| Training dataset | 525 Bird Species (~90k images) |
| Training epochs | 15 |
| Optimizer | AdamW with cosine LR decay |
| Effective batch size | 64 |

**Next step:** Download `vit_birds_finetuned.pth` and `bird_class_names.json` from your Google Drive and place them in the `EyeM/models/` directory to use in the Streamlit app.